In [1]:
import pandas as pd 
import os 
import sys

sys.path.append(os.path.abspath(os.path.join(os.getcwd() , os.pardir)))

from utils.helpers import get_db_engine
from utils.loggers_config import logger

engine = get_db_engine()

query = "SELECT * FROM engineered_churn_data"

try:
    logger.info("Attempting to fetch data from PostgreSQL...")
    df = pd.read_sql(query, engine)
    logger.info(f"Data ingestion successful! Loaded {df.shape[0]} rows and {df.shape[1]} columns.")
except Exception as e:
    logger.error(f"Failed to ingest data. Error: {str(e)}")
    raise e

df.head()

[2026-05-11 18:22:09,671: INFO: 4007626463: Attempting to fetch data from PostgreSQL...]
[2026-05-11 18:22:12,504: INFO: 4007626463: Data ingestion successful! Loaded 440832 rows and 18 columns.]


,Age,Tenure,Usage Frequency,Support Calls,Payment Delay,Contract Length,Total Spend,Last Interaction,Churn,Gender_male,Subscription Type_premium,Subscription Type_standard,is_critical_payment_delay,is_low_spender,high_support_risk,is_passive_user,is_low_usage,tenure_segment
0,47,38.0,12.0,10,8,0,250.0,22.0,1,False,False,True,0,1,1,1,0,4
1,39,2.0,27.0,10,24,0,348.0,1.0,1,False,False,False,1,1,1,0,0,1
2,60,21.0,19.0,7,11,0,718.0,16.0,1,True,True,False,0,0,1,1,0,3
3,22,35.0,25.0,8,22,0,805.0,8.0,1,False,True,False,1,0,1,0,0,4
4,19,14.0,16.0,6,14,0,776.0,9.0,1,False,False,True,0,0,1,0,0,3


In [5]:
query = "SELECT * FROM engineered_churn_data"

try:
    logger.info("Attempting to fetch data from PostgreSQL...")
    df = pd.read_sql(query, engine)
    
    logger.info(f"Data ingestion successful! Loaded {df.shape[0]} rows and {df.shape[1]} columns.")
except Exception as e:
    logger.error(f"Failed to ingest data. Error: {str(e)}")
    raise e

[2026-05-11 18:33:42,542: INFO: 1955847757: Attempting to fetch data from PostgreSQL...]
[2026-05-11 18:33:45,056: INFO: 1955847757: Data ingestion successful! Loaded 440832 rows and 18 columns.]


In [ ]:
from sklearn.model_selection import train_test_split  
X = df.drop('Churn', axis=1)
y = df['Churn']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
logger.info(f"Split complete. Training: {X_train.shape[0]} rows, Validation: {X_val.shape[0]} rows.")

[2026-05-11 18:35:21,211: INFO: 2529232317: Split complete. Training: 352665 rows, Validation: 88167 rows.]


In [8]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
import yaml
import mlflow
import mlflow.xgboost

with open("../config/hyperparams.yaml", "r") as f:
    config = yaml.safe_load(f)
    params = config['xgboost']

mlflow.set_experiment("Customer_Churn_XGBoost")

with mlflow.start_run():
    mlflow.log_params(params)
    
    model = XGBClassifier(**params)
    
    logger.info("Training with MLflow tracking...")

    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_val)
    
    acc = accuracy_score(y_val, y_pred)
    mlflow.log_metric("accuracy", acc)
    
    mlflow.xgboost.log_model(model, "model")
    
    logger.info(f"Training logged to MLflow. Accuracy: {acc}")

[2026-05-11 19:28:39,353: WARNING: connectionpool: Retrying (Retry(total=6, connect=6, read=7, redirect=7, status=7)) after connection broken by 'NewConnectionError("HTTPConnection(host='localhost', port=5000): Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it")': /api/2.0/mlflow/experiments/get-by-name?experiment_name=Customer_Churn_XGBoost]
[2026-05-11 19:28:48,085: WARNING: connectionpool: Retrying (Retry(total=5, connect=5, read=7, redirect=7, status=7)) after connection broken by 'NewConnectionError("HTTPConnection(host='localhost', port=5000): Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it")': /api/2.0/mlflow/experiments/get-by-name?experiment_name=Customer_Churn_XGBoost]
[2026-05-11 19:29:01,089: WARNING: connectionpool: Retrying (Retry(total=4, connect=4, read=7, redirect=7, status=7)) after connection broken by 'NewC

2026/05/11 19:30:01 INFO mlflow.tracking.fluent: Experiment with name 'Customer_Churn_XGBoost' does not exist. Creating a new experiment.


[2026-05-11 19:30:02,938: INFO: 32299237: Training with MLflow tracking...]


c:\Users\ASUS\customer-churn-mlops-pipeline\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [19:30:04] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
2026/05/11 19:30:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/11 19:30:21 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


[2026-05-11 19:30:24,189: INFO: 32299237: Training logged to MLflow. Accuracy: 0.9878299136865267]
🏃 View run angry-elk-545 at: http://localhost:5000/#/experiments/1/runs/b9de4c7f94804c5eb059c8e8d1ce43f3
🧪 View experiment at: http://localhost:5000/#/experiments/1
